In [10]:
import numpy as np
import copy

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dsigmoid(y):
    return y * (1 - y)

In [11]:
class MLP_2_2_1:
    def __init__(self, W_input_hidden, b_hidden, W_hidden_output, b_out, lr=0.5):
        self.W1 = np.array(W_input_hidden, dtype=float)
        self.b1 = np.array(b_hidden, dtype=float).reshape(-1)
        self.W2 = np.array(W_hidden_output, dtype=float).reshape(1, -1)
        self.b2 = float(b_out)
        self.lr = lr

    def forward(self, x):
        self.x = np.array(x, dtype=float).reshape(-1)
        self.z1 = self.W1.dot(self.x) + self.b1 
        self.a1 = sigmoid(self.z1)              
        self.z2 = self.W2.dot(self.a1) + self.b2
        self.a2 = sigmoid(self.z2)[0]
        return self.a2

    def backward(self, target):
        # compute gradients and update weights 
        error = target - self.a2
        
        delta2 = error * dsigmoid(self.a2)  
        dW2 = delta2 * self.a1  
        db2 = delta2
        
        delta1 = (self.W2.T.flatten() * delta2) * dsigmoid(self.a1)
        
        dW1 = np.outer(delta1, self.x)
        db1 = delta1

        # update weights
        self.W2 += self.lr * dW2.reshape(self.W2.shape)
        self.b2 += self.lr * db2
        self.W1 += self.lr * dW1
        self.b1 += self.lr * db1

        return error**2

    def predict(self, X):
        return np.array([self.forward(x) for x in X])

In [12]:
def train_stochastic(model, X, Y, max_epochs=5, tol=1e-8):
    # online learning: update after each example
    mse_history = []
    for epoch in range(max_epochs):
        sq_errs = []
        # iterate through examples (stochastic)
        for x, y in zip(X, Y):
            model.forward(x)
            sq_err = model.backward(y)
            sq_errs.append(sq_err)
        mse = np.mean(sq_errs)
        mse_history.append(mse)
        # convergence check (simple)
        if epoch>0 and abs(mse_history[-2]-mse_history[-1])<tol:
            break
    return mse_history

In [13]:
def train_batch(model, X, Y, max_epochs=5, tol=1e-8):
    # full-batch: accumulate gradients across examples then update
    mse_history = []
    for epoch in range(max_epochs):
        total_sq_err = 0.0
        # store gradients
        dW2_acc = np.zeros_like(model.W2)
        db2_acc = 0.0
        dW1_acc = np.zeros_like(model.W1)
        db1_acc = np.zeros_like(model.b1)
        for x, y in zip(X, Y):
            # forward
            out = model.forward(x)
            err = y - out
            total_sq_err += err**2
            delta2 = err * dsigmoid(model.a2)
            dW2 = delta2 * model.a1
            db2 = delta2
            delta1 = (model.W2.T.flatten() * delta2) * dsigmoid(model.a1)
            dW1 = np.outer(delta1, model.x)
            db1 = delta1
            dW2_acc += dW2.reshape(model.W2.shape)
            db2_acc += db2
            dW1_acc += dW1
            db1_acc += db1
        # average gradients
        n = len(X)
        model.W2 += model.lr * (dW2_acc / n)
        model.b2 += model.lr * (db2_acc / n)
        model.W1 += model.lr * (dW1_acc / n)
        model.b1 += model.lr * (db1_acc / n)
        mse = total_sq_err / n
        mse_history.append(mse)
        if epoch>0 and abs(mse_history[-2]-mse_history[-1])<tol:
            break
    return mse_history

In [14]:
def train_minibatch(model, X, Y, batch_size=2, max_epochs=5, tol=1e-8):
    mse_history = []
    n = len(X)
    for epoch in range(max_epochs):
        total_sq_err = 0.0
        idx = np.arange(n)
        np.random.shuffle(idx)
        for i in range(0, n, batch_size):
            batch_idx = idx[i:i+batch_size]
            dW2_acc = np.zeros_like(model.W2)
            db2_acc = 0.0
            dW1_acc = np.zeros_like(model.W1)
            db1_acc = np.zeros_like(model.b1)
            for j in batch_idx:
                x = X[j]
                y = Y[j]
                out = model.forward(x)
                err = y - out
                total_sq_err += err**2
                delta2 = err * dsigmoid(model.a2)
                dW2 = delta2 * model.a1
                db2 = delta2
                delta1 = (model.W2.T.flatten() * delta2) * dsigmoid(model.a1)
                dW1 = np.outer(delta1, model.x)
                db1 = delta1
                dW2_acc += dW2.reshape(model.W2.shape)
                db2_acc += db2
                dW1_acc += dW1
                db1_acc += db1
            m = len(batch_idx)
            model.W2 += model.lr * (dW2_acc / m)
            model.b2 += model.lr * (db2_acc / m)
            model.W1 += model.lr * (dW1_acc / m)
            model.b1 += model.lr * (db1_acc / m)
        mse = total_sq_err / n
        mse_history.append(mse)
        if epoch>0 and abs(mse_history[-2]-mse_history[-1])<tol:
            break
    return mse_history

In [15]:
X = np.array([
    [1.0, 1.5],
    [0.2, 0.5],
    [0.75, 0.6],
    [0.6, 0.8]
])
Y = np.array([1.2, 0.8, 1.8, 0.5])

In [16]:
W1_init = [[0.1, -0.2], [0.4, 0.2]]
b1_init = [0.0, 0.0]

W2_init = [[-0.3, 0.2]]
b2_init = 0.0

lr = 0.5

In [17]:
# Run stochastic training
model_stoch = MLP_2_2_1(copy.deepcopy(W1_init), copy.deepcopy(b1_init), copy.deepcopy(W2_init), b2_init, lr=lr)
mses_stoch = train_stochastic(model_stoch, X, Y, max_epochs=5)

# Full-batch training
model_batch = MLP_2_2_1(copy.deepcopy(W1_init), copy.deepcopy(b1_init), copy.deepcopy(W2_init), b2_init, lr=lr)
mses_batch = train_batch(model_batch, X, Y, max_epochs=5)

# Mini-batch training (batch size 2)
model_minib = MLP_2_2_1(copy.deepcopy(W1_init), copy.deepcopy(b1_init), copy.deepcopy(W2_init), b2_init, lr=lr)
mses_minib = train_minibatch(model_minib, X, Y, batch_size=2, max_epochs=5)

In [18]:
# Replace the printing section in Part A with this shorter format:
print("PART A - Final results :-\n")
print(f"Stochastic final MSE: {mses_stoch[-1]:.4f}")
print("Stochastic final weights W1:\n", np.round(model_stoch.W1, 4))
print("Stochastic final biases b1:\n", np.round(model_stoch.b1, 4))
print("Stochastic final W2:\n", np.round(model_stoch.W2, 4))
print(f"Stochastic final b2: {model_stoch.b2:.4f}\n")

print(f"Batch final MSE: {mses_batch[-1]:.4f}")
print("Batch final weights W1:\n", np.round(model_batch.W1, 4))
print("Batch final biases b1:\n", np.round(model_batch.b1, 4))
print("Batch final W2:\n", np.round(model_batch.W2, 4))
print(f"Batch final b2: {model_batch.b2:.4f}\n")

print(f"Mini-batch final MSE: {mses_minib[-1]:.4f}")
print("Mini-batch final weights W1:\n", np.round(model_minib.W1, 4))
print("Mini-batch final biases b1:\n", np.round(model_minib.b1, 4))
print("Mini-batch final W2:\n", np.round(model_minib.W2, 4))
print(f"Mini-batch final b2: {model_minib.b2:.4f}")

PART A - Final results :-

Stochastic final MSE: 0.3266
Stochastic final weights W1:
 [[ 0.0789 -0.2251]
 [ 0.4687  0.2708]]
Stochastic final biases b1:
 [-0.027   0.0853]
Stochastic final W2:
 [[0.0988 0.746 ]]
Stochastic final b2: 0.8575

Batch final MSE: 0.4605
Batch final weights W1:
 [[ 0.0854 -0.2165]
 [ 0.416   0.2178]]
Batch final biases b1:
 [-0.0192  0.0212]
Batch final W2:
 [[-0.1476  0.4006]]
Batch final b2: 0.3235

Mini-batch final MSE: 0.3928
Mini-batch final weights W1:
 [[ 0.0794 -0.223 ]
 [ 0.4347  0.2377]]
Mini-batch final biases b1:
 [-0.0266  0.0451]
Mini-batch final W2:
 [[-0.0364  0.5522]]
Mini-batch final b2: 0.5628
